In [ ]:
import os
import sys
import pandas as pd
from datetime import datetime

from src.nba_scrapping import *
from src.utils import *
from src.config import *


In [ ]:
start_time = datetime.now()
print("Start:", start_time)

# 🔍 Trouver le dernier dossier de boxscores scrap


In [ ]:

last_dir = get_latest_dir_child(DATA_LAST_BOXSCORES_BATCHES_DIR)
if last_dir is None:
    raise FileNotFoundError(f"❌ Aucun dossier trouvé dans {DATA_LAST_BOXSCORES_BATCHES_DIR}")

season_dirs = os.listdir(last_dir)
if len(season_dirs) != 1:
    raise ValueError(f"❌ Un seul dossier de saison attendu, trouvé : {season_dirs}")

season_name = season_dirs[0]
season_path = os.path.join(last_dir, season_name)
merged_path = os.path.join(season_path, "merged_batches")

# ⛔ Check si déjà traité
final_exists = any(f.startswith("final_merged_all_boxscores_") and f.endswith(".csv") for f in os.listdir(merged_path)) if os.path.exists(merged_path) else False

if final_exists:
    print(f"✅ Le dossier de la saison {season_path} scrappé et clean dans {merged_path} a déjà été traité. Fin du script.")
    sys.exit(0)

# 🔁 Retry erreurs éventuelles sur les endpoints


In [ ]:

print(f"🔁 Retry des endpoints échoués pour la saison {season_name}")
retry_failed_boxscores_for_season(season_path, max_retries=10)


# 🔄 Merge des endpoints en CSV uniques


In [ ]:
print(f"📦 Fusion des endpoints de boxscores pour {season_name}")
merge_boxscore_batches_for_season(season_path)


# 📚 Merge complet final


In [ ]:

run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
print(f"📄 Fusion complète des endpoints en un seul fichier final pour {season_name}")
final_output_path = merge_all_boxscore_stats(merged_path, output_filename=f"final_merged_all_boxscores_{run_timestamp}.csv")


# 📊 Charger le dernier fichier de boxscores merged complet (historique)


In [ ]:
print("\n📦 Chargement du fichier merged historique le plus récent...")
last_merged_hist_path = get_latest_file(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR)

if last_merged_hist_path is None:
    raise FileNotFoundError("❌ Aucun fichier historique merged trouvé dans DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR")

historical_df = pd.read_csv(last_merged_hist_path, dtype={'gameId': str})
print(f"✅ Chargé : {last_merged_hist_path} ({len(historical_df)} lignes)")


# 🔀 Ajout du fichier traité à l'historique complet


In [ ]:
print(f"📎 Concaténation avec les derniers boxscores scrapés ({season_name})")
latest_df = pd.read_csv(final_output_path, dtype={'gameId': str})

merged_df = pd.concat([historical_df, latest_df], ignore_index=True)

# 💾 Sauvegarde globale
os.makedirs(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR, exist_ok=True)
merged_global_path = os.path.join(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR, f"all_seasons_boxscores_merged_{run_timestamp}.csv")
merged_df.to_csv(merged_global_path, index=False)

print(f"✅ Dataset final sauvegardé : {merged_global_path}")
print("End:", datetime.now())
print("Duration:", datetime.now() - start_time)
